# 🔬 Sparse Autoencoders (SAEs): A Deep Dive
### *From the Superposition Problem to Mechanistic Interpretability*

---

> *"The most important question in mechanistic interpretability is: what are the fundamental units of computation in neural networks?"*  
> — Neel Nanda

---

## 📌 What You Will Learn

This notebook is a complete, self-contained deep dive into **Sparse Autoencoders (SAEs)** — currently the most important tool in mechanistic interpretability research. By the end, you will:

1. Understand **why** SAEs exist — the superposition problem
2. Understand the **mathematical formulation** of SAEs
3. **Implement** an SAE from scratch in PyTorch
4. Train an SAE on a **toy superposition model**
5. **Visualize** what features the SAE learns
6. Understand how this connects to **real interpretability research**

---

## 🗺️ Table of Contents

1. [Background: The Fundamental Problem](#section-1)
2. [What is Superposition?](#section-2)
3. [The Sparse Autoencoder: Intuition](#section-3)
4. [Mathematical Formulation](#section-4)
5. [Setup & Imports](#section-5)
6. [Part 1: Building the Toy Superposition Model](#section-6)
7. [Part 2: Implementing the Sparse Autoencoder](#section-7)
8. [Part 3: Training the SAE](#section-8)
9. [Part 4: Evaluating & Visualizing Learned Features](#section-9)
10. [Part 5: The Loss Landscape — Sparsity vs Reconstruction](#section-10)
11. [Part 6: Dead Neurons & How to Fix Them](#section-11)
12. [How SAEs Are Used in Real Research](#section-12)
13. [Limitations & Open Problems](#section-13)
14. [Further Reading & Resources](#section-14)

---
<a id='section-1'></a>
## 1. Background: The Fundamental Problem

### Why is Interpretability Hard?

When a large language model processes text, billions of floating-point numbers (activations) flow through it. Each neuron fires with some value. The central challenge of **mechanistic interpretability** is:

> **What does each neuron represent? What is it "thinking about"?**

The naive hope is: *one neuron = one concept*. For example:
- Neuron 42 fires whenever it sees the word "banana"
- Neuron 1337 fires whenever it detects a question
- Neuron 99 fires for French text

This is called **monosemanticity** — one neuron, one meaning. If neurons were monosemantic, interpretability would be easy: just measure what makes each neuron fire.

### The Reality: Polysemanticity

Early interpretability researchers discovered something disturbing: **neurons are polysemantic**.

A single neuron might activate for:
- *Cats, curved shapes, and the word "arc"*
- *Violence in text AND a specific type of Python error*
- *Multiple unrelated programming concepts simultaneously*

This was documented in Anthropic's landmark 2022 paper ["Toy Models of Superposition"](https://transformer-circuits.pub/2022/toy_model/index.html).

But **why** does this happen? It's not a design bug. It's a *mathematical inevitability* — and understanding it is the key to understanding SAEs.

---
<a id='section-2'></a>
## 2. What is Superposition?

### The Dimensionality Mismatch

Consider a language model with a hidden dimension of **d = 512**. This means each layer's activations live in a 512-dimensional space. But the world has **millions of features** — concepts, facts, patterns — that a capable model must represent.

**How can you represent 1,000,000 features in 512 dimensions?**

The answer: **you can't represent them all simultaneously, orthogonally.** In a 512-dimensional space, you can have at most 512 perfectly orthogonal (non-interfering) directions.

But here's the key insight from information theory and compressed sensing:

> **If features are sparse** (only a few are active at any given time), you can pack FAR more than d features into d dimensions — they will just interfere slightly with each other.

### The Mathematical Intuition

Suppose you have `n` features but only `d` neurons (n >> d). You encode feature `i` as a direction $\mathbf{e}_i \in \mathbb{R}^d$.

A model activation $\mathbf{x}$ is then:

$$\mathbf{x} = \sum_{i} f_i \cdot \mathbf{e}_i$$

where $f_i$ is the "activation value" of feature $i$.

If only $k$ features are active at once (sparsity), then the **interference** between features is small — proportional to $k/n$. As long as $k << n$, the model can approximately reconstruct which features were active.

This is **superposition**: packing n features into d < n dimensions, exploiting sparsity.

### Why This Breaks Interpretability

In superposition, **a single neuron participates in representing multiple features**. You cannot look at neuron 42 in isolation and say what it means — it's a mixture. This is the root cause of polysemanticity.

### The Geometry: Almost-Orthogonal Vectors

In high dimensions, you can pack many more "almost orthogonal" vectors than truly orthogonal ones. This is the **Johnson-Lindenstrauss phenomenon**.

In $\mathbb{R}^d$, you can fit:
- **d** perfectly orthogonal vectors
- **Exponentially many** vectors with pairwise dot products $|\mathbf{e}_i \cdot \mathbf{e}_j| < \epsilon$

For example, in $\mathbb{R}^{512}$, you can pack thousands of near-orthogonal vectors. Each represents a different feature. The model exploits this to compress its world-knowledge.

---
<a id='section-3'></a>
## 3. The Sparse Autoencoder: Intuition

### The Core Idea

If a model's activations are a **compressed, superimposed mixture of features**, can we decompress them?

A **Sparse Autoencoder** is a neural network trained to:
1. **Encode** model activations $\mathbf{x} \in \mathbb{R}^d$ into a **larger, sparse** representation $\mathbf{f} \in \mathbb{R}^n$ (where $n >> d$)
2. **Decode** that sparse representation back to reconstruct $\mathbf{x}$

The key constraint: $\mathbf{f}$ must be **sparse** — most of its entries must be zero or near-zero.

### Why Sparsity = Features

The sparsity constraint forces the SAE to find a representation where:
- Each "slot" in $\mathbf{f}$ is only active for a small, specific set of inputs
- Each slot represents a single, coherent concept (monosemantic!)

The SAE essentially **reverses the compression** the model applied, recovering the underlying sparse feature basis.

### The Architecture

```
Input activations x ∈ ℝᵈ
        │
        ▼
┌───────────────────┐
│  ENCODER          │   W_enc ∈ ℝⁿˣᵈ, b_enc ∈ ℝⁿ
│  f = ReLU(W_enc·x + b_enc) │
└───────────────────┘
        │
        ▼
Sparse features f ∈ ℝⁿ  (n >> d, most entries = 0)
        │
        ▼
┌───────────────────┐
│  DECODER          │   W_dec ∈ ℝᵈˣⁿ, b_dec ∈ ℝᵈ
│  x̂ = W_dec·f + b_dec  │
└───────────────────┘
        │
        ▼
Reconstructed activations x̂ ∈ ℝᵈ
```

The decoder weight matrix $W_{dec}$ is the **feature dictionary**: each column is a "feature direction" in activation space.

---
<a id='section-4'></a>
## 4. Mathematical Formulation

### The SAE Forward Pass

Given an input activation $\mathbf{x} \in \mathbb{R}^d$:

**Step 1: Pre-bias subtraction**  
$$\mathbf{x}' = \mathbf{x} - \mathbf{b}_{dec}$$

This centers the input around the decoder bias (important for the encoder to learn meaningful directions).

**Step 2: Encoder**
$$\mathbf{f} = \text{ReLU}(W_{enc} \mathbf{x}' + \mathbf{b}_{enc})$$

where $W_{enc} \in \mathbb{R}^{n \times d}$, $\mathbf{b}_{enc} \in \mathbb{R}^n$, and $n >> d$.

The ReLU enforces **non-negativity** and contributes to sparsity.

**Step 3: Decoder**
$$\hat{\mathbf{x}} = W_{dec} \mathbf{f} + \mathbf{b}_{dec}$$

where $W_{dec} \in \mathbb{R}^{d \times n}$.

Each column $\mathbf{d}_i$ of $W_{dec}$ is a **feature direction** (normalized to unit length).

### The Loss Function

$$\mathcal{L} = \underbrace{\|\mathbf{x} - \hat{\mathbf{x}}\|_2^2}_{\text{Reconstruction Loss}} + \lambda \underbrace{\|\mathbf{f}\|_1}_{\text{Sparsity Penalty}}$$

**Reconstruction Loss** ($L_2$): The SAE must faithfully reconstruct the original activations. Without this, the SAE could just output zeros.

**Sparsity Penalty** ($L_1$): The $L_1$ norm encourages sparsity. Why $L_1$ and not $L_2$?

- $L_2$ penalty: $\sum f_i^2$ — pushes all values small, but rarely to exactly zero
- $L_1$ penalty: $\sum |f_i|$ — has a sharp corner at zero, induces **exact zeros** (sparse solutions)

**$\lambda$** is a hyperparameter balancing:
- High $\lambda$ → very sparse, but poor reconstruction
- Low $\lambda$ → good reconstruction, but not sparse (not interpretable)

### The Decoder Normalization Constraint

The decoder columns $\mathbf{d}_i$ are constrained to have **unit norm**:
$$\|\mathbf{d}_i\|_2 = 1 \quad \forall i$$

Without this, the model can cheat: it can make $W_{dec}$ columns very small (absorbing the features into large $\mathbf{f}$ values) and still reconstruct well while appearing sparse. The normalization removes this degeneracy.

### What the SAE is Really Doing

The SAE is solving a **sparse dictionary learning** problem:

> Given observations $\{\mathbf{x}^{(1)}, ..., \mathbf{x}^{(T)}\}$, find a dictionary $D \in \mathbb{R}^{d \times n}$ and sparse codes $\{\mathbf{f}^{(t)}\}$ such that $\mathbf{x}^{(t)} \approx D \mathbf{f}^{(t)}$ and $\mathbf{f}^{(t)}$ is sparse.

This is closely related to **Independent Component Analysis (ICA)**, **ISTA** (Iterative Shrinkage-Thresholding Algorithm), and compressed sensing — all classical signal processing ideas applied to neural network internals.

---
<a id='section-5'></a>
## 5. Setup & Imports

In [ ]:
# Install dependencies (run this cell first in Colab)
!pip install torch numpy matplotlib seaborn einops --quiet

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from dataclasses import dataclass
from typing import Optional, Tuple
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

# ── Reproducibility ──────────────────────────────────────────────
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# ── Device selection ─────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ── Dark GitHub-style plotting theme ────────────────────────────
plt.style.use('dark_background')
COLORS = {
    'blue':   '#58a6ff',
    'green':  '#3fb950',
    'orange': '#d29922',
    'red':    '#f85149',
    'purple': '#bc8cff',
    'cyan':   '#39d0d0',
    'text':   '#e6edf3',
    'grid':   '#21262d',
    'bg':     '#0d1117',
    'panel':  '#161b22',
}

def styled_ax(ax, title='', xlabel='', ylabel=''):
    """Apply consistent dark styling to a matplotlib axis."""
    ax.set_facecolor(COLORS['panel'])
    ax.tick_params(colors=COLORS['text'], labelsize=9)
    ax.xaxis.label.set_color(COLORS['text'])
    ax.yaxis.label.set_color(COLORS['text'])
    ax.title.set_color(COLORS['text'])
    for spine in ax.spines.values():
        spine.set_edgecolor(COLORS['grid'])
    ax.grid(True, color=COLORS['grid'], linewidth=0.5, linestyle='--', alpha=0.7)
    if title:  ax.set_title(title, fontsize=11, fontweight='bold', color=COLORS['text'])
    if xlabel: ax.set_xlabel(xlabel, fontsize=9)
    if ylabel: ax.set_ylabel(ylabel, fontsize=9)
    return ax

print('All imports successful! Ready to explore Sparse Autoencoders.')

---
<a id='section-6'></a>
## 6. Part 1: Building the Toy Superposition Model

### The Setup

We will recreate the key experiment from Anthropic's "Toy Models of Superposition" paper.

**The model:**
- Has `n_features` input features (e.g., 20)
- Compresses them into `d_hidden` dimensions (e.g., 4)
- Then decompresses back to `n_features`

Since `n_features >> d_hidden`, the model is forced to use superposition.

**The data:**
- Each input is a vector where each feature is independently active with probability `p` (feature frequency)
- Features have varying **importance** — some matter more than others

This is an idealized version of what happens inside a real LLM, where tokens activate sparse, varying-frequency concepts.

In [ ]:
@dataclass
class ToyModelConfig:
    """Configuration for the toy superposition model."""
    n_features: int = 20        # Number of ground-truth features in the world
    d_hidden:   int = 4         # Hidden dim (bottleneck) — causes superposition
    n_samples:  int = 100_000   # Training samples
    batch_size: int = 1024
    n_epochs:   int = 1000
    lr:         float = 1e-3
    sparsity:   float = 0.05    # Probability each feature is active per sample

cfg = ToyModelConfig()
print(f'Toy Model Config:')
print(f'  {cfg.n_features} features compressed into {cfg.d_hidden} dimensions')
print(f'  Compression ratio: {cfg.n_features / cfg.d_hidden:.1f}x')
print(f'  Feature sparsity: {cfg.sparsity:.0%} active per sample on average')

In [ ]:
def generate_sparse_data(cfg: ToyModelConfig) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Generate sparse synthetic feature data.

    Each sample is a vector of n_features values.
    Each feature is active (nonzero) with probability `sparsity`,
    and when active, drawn from a uniform distribution [0, 1].

    Features have geometric importance weighting:
        importance[i] = 0.9^i
    so feature 0 is most important, feature 19 least.

    Returns:
        data:       (n_samples, n_features) feature values
        importance: (n_features,) importance weights
    """
    # Mask: is feature i active in sample j?
    mask = (torch.rand(cfg.n_samples, cfg.n_features) < cfg.sparsity).float()

    # Feature values: uniform [0,1] when active, 0 otherwise
    values = torch.rand(cfg.n_samples, cfg.n_features)
    data = mask * values

    # Feature importance: geometric decay
    importance = torch.tensor([0.9 ** i for i in range(cfg.n_features)])

    return data.to(device), importance.to(device)


data, importance = generate_sparse_data(cfg)

# ── Visualize the data ───────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), facecolor=COLORS['bg'])
fig.suptitle('Toy Superposition Data', fontsize=14, color=COLORS['text'], fontweight='bold')

# Plot 1: Sample heatmap
ax = axes[0]
sample = data[:50].cpu().numpy()
im = ax.imshow(sample, aspect='auto', cmap='Blues', interpolation='nearest')
plt.colorbar(im, ax=ax)
styled_ax(ax, 'First 50 Samples', 'Feature Index', 'Sample Index')

# Plot 2: Feature activation frequency
ax = axes[1]
freq = (data > 0).float().mean(0).cpu().numpy()
ax.bar(range(cfg.n_features), freq, color=COLORS['blue'], alpha=0.8)
ax.axhline(cfg.sparsity, color=COLORS['red'], linestyle='--', linewidth=1.5, label=f'Expected ({cfg.sparsity:.0%})')
ax.legend(fontsize=8, facecolor=COLORS['panel'])
styled_ax(ax, 'Feature Activation Frequency', 'Feature Index', 'P(active)')

# Plot 3: Feature importance
ax = axes[2]
ax.bar(range(cfg.n_features), importance.cpu().numpy(), color=COLORS['orange'], alpha=0.8)
styled_ax(ax, 'Feature Importance (0.9^i decay)', 'Feature Index', 'Importance')

plt.tight_layout()
plt.show()

print(f'Data shape: {data.shape}')
print(f'Average features active per sample: {(data > 0).float().sum(1).mean().item():.2f}')

In [ ]:
class ToySuperpositonModel(nn.Module):
    """
    A minimal model that exhibits superposition.

    Architecture:
        x (n_features) -> W (d_hidden) -> ReLU -> W^T (n_features) -> ReLU -> x̂

    By using W^T as the decoder, we enforce a 'tied weights' structure,
    which mirrors what Anthropic studied in their toy model paper.
    The bias b is per-feature.

    The compressed representation h = W @ x lives in d_hidden dimensions.
    Since n_features >> d_hidden, the model must use superposition to
    represent all features.
    """
    def __init__(self, cfg: ToyModelConfig):
        super().__init__()
        self.cfg = cfg

        # W: the compression matrix — rows are feature directions in hidden space
        # Shape: (n_features, d_hidden)
        self.W = nn.Parameter(torch.randn(cfg.n_features, cfg.d_hidden) * 0.01)

        # Per-feature bias
        self.b = nn.Parameter(torch.zeros(cfg.n_features))

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """Compress: x (batch, n_features) -> h (batch, d_hidden)"""
        return x @ self.W  # (batch, d_hidden)

    def decode(self, h: torch.Tensor) -> torch.Tensor:
        """Decompress: h (batch, d_hidden) -> x_hat (batch, n_features)"""
        return F.relu(h @ self.W.T + self.b)  # (batch, n_features)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        h = self.encode(x)
        return self.decode(h)


def train_toy_model(cfg: ToyModelConfig, data: torch.Tensor, importance: torch.Tensor):
    """Train the toy superposition model and return trained model + loss history."""
    model = ToySuperpositonModel(cfg).to(device)
    optimizer = optim.Adam(model.parameters(), lr=cfg.lr)

    losses = []
    n_batches = cfg.n_samples // cfg.batch_size

    for epoch in range(cfg.n_epochs):
        epoch_loss = 0.0
        perm = torch.randperm(cfg.n_samples, device=device)

        for i in range(n_batches):
            idx = perm[i * cfg.batch_size:(i + 1) * cfg.batch_size]
            x_batch = data[idx]  # (batch, n_features)

            optimizer.zero_grad()
            x_hat = model(x_batch)

            # Importance-weighted MSE loss
            # We care more about getting high-importance features right
            loss = ((x_batch - x_hat) ** 2 * importance).mean()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item()

        avg_loss = epoch_loss / n_batches
        losses.append(avg_loss)

        if epoch % 200 == 0:
            print(f'  Epoch {epoch:4d} | Loss: {avg_loss:.6f}')

    return model, losses


print('Training toy superposition model...')
toy_model, toy_losses = train_toy_model(cfg, data, importance)
print('Done!')

In [ ]:
def visualize_superposition(model: ToySuperpositonModel, cfg: ToyModelConfig):
    """
    Visualize the superposition structure learned by the toy model.

    We plot the weight matrix W (n_features, d_hidden).
    Each row is a feature's 'direction' in the d_hidden-dimensional space.

    If d_hidden = 2, we can plot these directions as 2D vectors.
    Interference (non-orthogonality) reveals superposition.
    """
    W = model.W.detach().cpu().numpy()  # (n_features, d_hidden)

    # Compute interference matrix: W @ W^T (n_features, n_features)
    # Diagonal = feature norms, off-diagonal = interference
    interference = W @ W.T

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=COLORS['bg'])
    fig.suptitle('Superposition: How Features Are Packed into Hidden Space',
                 fontsize=13, color=COLORS['text'], fontweight='bold')

    # Plot 1: Feature direction norms
    ax = axes[0]
    norms = np.linalg.norm(W, axis=1)
    colors = [COLORS['blue'] if n > 0.5 else COLORS['orange'] for n in norms]
    ax.bar(range(cfg.n_features), norms, color=colors, alpha=0.85)
    ax.axhline(1.0, color=COLORS['green'], linestyle='--', linewidth=1.5, label='Full representation')
    ax.axhline(0.5, color=COLORS['red'], linestyle=':', linewidth=1.5, label='Half representation')
    ax.legend(fontsize=8, facecolor=COLORS['panel'])
    styled_ax(ax, 'Feature Representation Norms',
              'Feature Index', '||W[i]|| (representation strength)')

    # Plot 2: Interference matrix
    ax = axes[1]
    mask_diag = np.eye(cfg.n_features, dtype=bool)
    disp = np.abs(interference)
    np.fill_diagonal(disp, 0)  # Zero out diagonal for cleaner visualization
    im = ax.imshow(disp, cmap='hot', aspect='auto')
    plt.colorbar(im, ax=ax, label='|interference|')
    styled_ax(ax, 'Feature Interference Matrix\n(off-diagonal |W_i · W_j|)',
              'Feature j', 'Feature i')

    # Plot 3: Training loss curve
    ax = axes[2]
    ax.plot(toy_losses, color=COLORS['blue'], linewidth=1.5, alpha=0.9)
    ax.set_yscale('log')
    styled_ax(ax, 'Training Loss (log scale)', 'Epoch', 'MSE Loss (weighted)')

    plt.tight_layout()
    plt.show()

    print(f'\nSuperposition Analysis:')
    print(f'  Features with strong representation (norm > 0.5): {(norms > 0.5).sum()}')
    print(f'  Features with weak/no representation (norm ≤ 0.5): {(norms <= 0.5).sum()}')
    mean_interference = np.abs(interference[~mask_diag]).mean()
    print(f'  Mean off-diagonal interference: {mean_interference:.4f}')
    print(f'  (0 = perfectly orthogonal features, higher = more superposition)')


visualize_superposition(toy_model, cfg)

### 🔍 What to Look For

- **Norm plot**: High-importance features (low index) tend to have larger norms — the model dedicates more of its limited space to them. Low-importance features may be "squeezed out" or stored in superposition.
- **Interference matrix**: Non-zero off-diagonal entries mean features are NOT orthogonal — they interfere with each other. This IS superposition.
- **The key point**: If you look at any single neuron's activations (any row of `W`), it will have contributions from multiple features. This is why individual neurons are polysemantic.

Now, we have a model that exhibits superposition. Our goal is to train an SAE to **decompose** its hidden activations back into individual features.

---
<a id='section-7'></a>
## 7. Part 2: Implementing the Sparse Autoencoder

Now we implement the SAE itself. The SAE will be trained on the **hidden representations** of the toy model — not the raw features, but the compressed, superposed activations inside the toy model.

In [ ]:
@dataclass
class SAEConfig:
    """Configuration for the Sparse Autoencoder."""
    d_in:         int   = 4      # Input dimension (= toy model's d_hidden)
    d_sae:        int   = 40     # SAE hidden dim — should be >> d_in
                                  # Expansion factor = d_sae / d_in = 10x
    l1_coeff:     float = 3e-4   # Lambda: sparsity penalty strength
    lr:           float = 2e-3   # Learning rate
    n_epochs:     int   = 2000   # Training epochs
    batch_size:   int   = 1024
    normalize_decoder: bool = True  # Enforce unit norm on decoder columns

sae_cfg = SAEConfig(d_in=cfg.d_hidden, d_sae=cfg.d_hidden * 10)

print(f'SAE Configuration:')
print(f'  Input dim (d_in):    {sae_cfg.d_in}')
print(f'  SAE dim (d_sae):     {sae_cfg.d_sae}')
print(f'  Expansion factor:    {sae_cfg.d_sae // sae_cfg.d_in}x')
print(f'  Sparsity coeff (λ):  {sae_cfg.l1_coeff}')
print(f'\nThe SAE will try to discover {sae_cfg.d_sae} features')
print(f'from activations living in {sae_cfg.d_in} dimensions.')
print(f'Ground truth: {cfg.n_features} features exist. SAE has {sae_cfg.d_sae} slots.')

In [ ]:
class SparseAutoencoder(nn.Module):
    """
    Sparse Autoencoder (SAE) — the core tool of mechanistic interpretability.

    Architecture:
        x ∈ ℝᵈ
        x' = x - b_dec                          (pre-bias centering)
        f  = ReLU(W_enc @ x' + b_enc)           (encoder, sparse features)
        x̂  = W_dec @ f + b_dec                 (decoder, reconstruction)

    Loss:
        L = ||x - x̂||² + λ ||f||₁

    Constraint:
        ||W_dec[:, i]||₂ = 1  for all i  (normalize decoder columns)

    Key implementation detail:
        W_enc and W_dec are separate parameters (NOT tied weights).
        In the original Anthropic SAE, they found that untied weights
        work better because the encoder and decoder can specialize.
    """

    def __init__(self, cfg: SAEConfig):
        super().__init__()
        self.cfg = cfg

        # ── Encoder ───────────────────────────────────────────────
        # W_enc: (d_in, d_sae) — each column is an encoder direction
        self.W_enc = nn.Parameter(
            nn.init.kaiming_uniform_(torch.empty(cfg.d_in, cfg.d_sae))
        )
        self.b_enc = nn.Parameter(torch.zeros(cfg.d_sae))

        # ── Decoder ───────────────────────────────────────────────
        # W_dec: (d_sae, d_in) — each row is a feature direction
        # We initialize as W_enc.T to start near a solution
        self.W_dec = nn.Parameter(
            nn.init.kaiming_uniform_(torch.empty(cfg.d_sae, cfg.d_in))
        )
        self.b_dec = nn.Parameter(torch.zeros(cfg.d_in))

        # Normalize decoder columns to unit norm at initialization
        if cfg.normalize_decoder:
            self._normalize_decoder()

        # Track dead neurons (features that never activate)
        self.register_buffer('feature_activation_counts', torch.zeros(cfg.d_sae))

    @torch.no_grad()
    def _normalize_decoder(self):
        """
        Normalize decoder rows to unit L2 norm.
        W_dec has shape (d_sae, d_in), so rows are the feature directions.
        """
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data = self.W_dec.data / norms

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        """
        Encoder forward pass.
        Args:
            x: (batch, d_in) activations
        Returns:
            f: (batch, d_sae) sparse feature activations
        """
        # Center around decoder bias first
        x_centered = x - self.b_dec
        # Linear projection + bias + ReLU
        pre_relu = x_centered @ self.W_enc + self.b_enc
        f = F.relu(pre_relu)
        return f

    def decode(self, f: torch.Tensor) -> torch.Tensor:
        """
        Decoder forward pass.
        Args:
            f: (batch, d_sae) sparse features
        Returns:
            x_hat: (batch, d_in) reconstructed activations
        """
        return f @ self.W_dec + self.b_dec

    def forward(self, x: torch.Tensor) -> dict:
        """
        Full SAE forward pass.
        Returns dict with all intermediate values for analysis.
        """
        f    = self.encode(x)
        x_hat = self.decode(f)

        # Reconstruction loss (L2)
        recon_loss = (x - x_hat).pow(2).sum(dim=-1).mean()

        # Sparsity penalty (L1 norm of feature activations)
        l1_loss = f.abs().sum(dim=-1).mean()

        # Total loss
        loss = recon_loss + self.cfg.l1_coeff * l1_loss

        # Update dead neuron tracking
        with torch.no_grad():
            self.feature_activation_counts += (f > 0).float().sum(dim=0)

        return {
            'loss':       loss,
            'recon_loss': recon_loss,
            'l1_loss':    l1_loss,
            'f':          f,
            'x_hat':      x_hat,
        }

    @property
    def dead_neurons(self) -> torch.Tensor:
        """Boolean mask of features that have never activated."""
        return self.feature_activation_counts == 0

    @property
    def n_dead(self) -> int:
        return self.dead_neurons.sum().item()


# Quick sanity check
sae = SparseAutoencoder(sae_cfg).to(device)
dummy_input = torch.randn(8, sae_cfg.d_in, device=device)
out = sae(dummy_input)
print('SAE sanity check:')
print(f'  Input shape:         {dummy_input.shape}')
print(f'  Feature shape:       {out["f"].shape}')
print(f'  Reconstruction shape:{out["x_hat"].shape}')
print(f'  Loss:                {out["loss"].item():.4f}')
print(f'  Recon loss:          {out["recon_loss"].item():.4f}')
print(f'  L1 loss:             {out["l1_loss"].item():.4f}')
print(f'  Sparsity:            {(out["f"] > 0).float().mean().item():.2%} features active')
print(f'  Decoder column norms: min={sae.W_dec.norm(dim=1).min():.4f}, max={sae.W_dec.norm(dim=1).max():.4f}')

---
<a id='section-8'></a>
## 8. Part 3: Training the SAE

We will train the SAE on the **hidden representations** extracted from our toy superposition model. The SAE sees only the compressed, superimposed activations — it must infer the original features.

In [ ]:
# Extract hidden activations from the toy model (what the SAE will train on)
toy_model.eval()
with torch.no_grad():
    hidden_acts = toy_model.encode(data)   # (n_samples, d_hidden)

print(f'Hidden activations shape: {hidden_acts.shape}')
print(f'Range: [{hidden_acts.min():.3f}, {hidden_acts.max():.3f}]')
print(f'Mean:  {hidden_acts.mean():.3f}')
print()
print('These are the superimposed activations the SAE will try to decompose.')
print('Each row is a {}-dim vector that encodes a mix of up to {} features.'.format(
    cfg.d_hidden, cfg.n_features))

# Visualize a few activation vectors
fig, ax = plt.subplots(figsize=(12, 3), facecolor=COLORS['bg'])
im = ax.imshow(hidden_acts[:30].cpu().numpy().T, aspect='auto', cmap='RdBu_r')
plt.colorbar(im, ax=ax, label='Activation value')
styled_ax(ax, 'Hidden Activations (first 30 samples)',
          'Sample Index', 'Hidden Dimension')
plt.tight_layout()
plt.show()

In [ ]:
def train_sae(
    sae: SparseAutoencoder,
    activations: torch.Tensor,
    cfg: SAEConfig,
) -> dict:
    """
    Train the SAE on a corpus of activations.

    Key training detail: after each gradient step, we re-normalize
    the decoder columns to unit norm. This prevents the model from
    absorbing feature magnitude into W_dec and bypassing the L1 penalty.

    Returns:
        Dictionary of training metrics over time.
    """
    optimizer = optim.Adam(sae.parameters(), lr=cfg.lr)

    n_samples = activations.shape[0]
    n_batches = n_samples // cfg.batch_size

    history = defaultdict(list)

    sae.train()

    for epoch in range(cfg.n_epochs):
        epoch_metrics = defaultdict(float)
        perm = torch.randperm(n_samples, device=device)

        for i in range(n_batches):
            idx  = perm[i * cfg.batch_size:(i + 1) * cfg.batch_size]
            x_batch = activations[idx]

            optimizer.zero_grad()
            out = sae(x_batch)
            out['loss'].backward()
            optimizer.step()

            # ── Re-normalize decoder after every gradient step ────
            if cfg.normalize_decoder:
                sae._normalize_decoder()

            # Track metrics
            epoch_metrics['loss']       += out['loss'].item()
            epoch_metrics['recon_loss'] += out['recon_loss'].item()
            epoch_metrics['l1_loss']    += out['l1_loss'].item()
            sparsity = (out['f'] > 0).float().mean().item()
            epoch_metrics['sparsity']   += sparsity

        for k, v in epoch_metrics.items():
            history[k].append(v / n_batches)
        history['dead_neurons'].append(sae.n_dead)

        if epoch % 400 == 0:
            print(f'Epoch {epoch:5d} | '
                  f'Loss: {history["loss"][-1]:.5f} | '
                  f'Recon: {history["recon_loss"][-1]:.5f} | '
                  f'L1: {history["l1_loss"][-1]:.5f} | '
                  f'Sparsity: {history["sparsity"][-1]:.2%} | '
                  f'Dead: {sae.n_dead}/{cfg.d_sae}')

    sae.eval()
    return dict(history)


print('Training SAE on toy model hidden activations...')
sae = SparseAutoencoder(sae_cfg).to(device)
history = train_sae(sae, hidden_acts, sae_cfg)
print('\nTraining complete!')

---
<a id='section-9'></a>
## 9. Part 4: Evaluating & Visualizing Learned Features

In [ ]:
def plot_training_curves(history: dict):
    """Plot comprehensive training metrics."""
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), facecolor=COLORS['bg'])
    fig.suptitle('SAE Training Metrics', fontsize=14, color=COLORS['text'], fontweight='bold')

    metrics = [
        ('loss',         'Total Loss',              COLORS['blue']),
        ('recon_loss',   'Reconstruction Loss',     COLORS['green']),
        ('l1_loss',      'L1 Sparsity Loss',        COLORS['orange']),
        ('sparsity',     'Feature Activation Rate', COLORS['purple']),
        ('dead_neurons', 'Dead Neurons Count',      COLORS['red']),
    ]

    for idx, (key, title, color) in enumerate(metrics):
        row, col = divmod(idx, 3)
        ax = axes[row][col]
        values = history[key]
        ax.plot(values, color=color, linewidth=1.5, alpha=0.9)
        if key == 'loss' or key == 'recon_loss':
            ax.set_yscale('log')
        styled_ax(ax, title, 'Epoch', key)

    # Hide unused subplot
    axes[1][2].set_visible(False)

    plt.tight_layout()
    plt.show()

    print('Final metrics:')
    for key, title, _ in metrics:
        val = history[key][-1]
        if key == 'sparsity':
            print(f'  {title}: {val:.2%}')
        else:
            print(f'  {title}: {val:.5f}')


plot_training_curves(history)

In [ ]:
def analyze_learned_features(sae: SparseAutoencoder, activations: torch.Tensor,
                              ground_truth_data: torch.Tensor, cfg_sae: SAEConfig,
                              cfg_toy: ToyModelConfig):
    """
    Analyze what features the SAE has learned.

    Key analyses:
    1. Feature activation frequencies
    2. Feature similarity matrix (are learned features orthogonal?)
    3. Correlation between SAE features and ground-truth features
    4. Reconstruction quality
    """
    sae.eval()

    with torch.no_grad():
        # Get SAE features for all samples
        out = sae(activations)
        features = out['f']          # (n_samples, d_sae)
        x_hat    = out['x_hat']      # (n_samples, d_in)

    features_np = features.cpu().numpy()
    acts_np     = activations.cpu().numpy()
    x_hat_np    = x_hat.cpu().numpy()
    gt_np       = ground_truth_data.cpu().numpy()

    # ── Feature statistics ────────────────────────────────────────
    activation_freq = (features_np > 0).mean(axis=0)   # (d_sae,)
    mean_activation  = features_np.mean(axis=0)         # (d_sae,)
    alive_mask       = activation_freq > 0.001          # Features that activate at least 0.1% of the time

    # ── Decoder feature similarity ────────────────────────────────
    W_dec_np = sae.W_dec.detach().cpu().numpy()          # (d_sae, d_in)
    similarity = W_dec_np @ W_dec_np.T                   # (d_sae, d_sae)

    # ── Correlation with ground truth ────────────────────────────
    # For each SAE feature, find the ground-truth feature it most correlates with
    n_alive = alive_mask.sum()
    alive_features = features_np[:, alive_mask]  # (n_samples, n_alive)

    # Compute correlation matrix between alive SAE features and GT features
    correlations = np.corrcoef(alive_features.T, gt_np.T)  # (n_alive + n_gt, n_alive + n_gt)
    sae_gt_corr  = correlations[:n_alive, n_alive:]  # (n_alive, n_gt)

    # Best match for each alive SAE feature
    best_gt_match    = np.abs(sae_gt_corr).argmax(axis=1)   # (n_alive,)
    best_gt_corr_val = np.abs(sae_gt_corr).max(axis=1)      # (n_alive,)

    # ── Reconstruction quality ────────────────────────────────────
    recon_error     = np.mean((acts_np - x_hat_np) ** 2)
    baseline_error  = np.mean(acts_np ** 2)  # MSE if we always predict 0
    variance_explained = 1 - recon_error / baseline_error

    # ── Visualizations ────────────────────────────────────────────
    fig, axes = plt.subplots(2, 3, figsize=(17, 9), facecolor=COLORS['bg'])
    fig.suptitle('SAE Feature Analysis', fontsize=14, color=COLORS['text'], fontweight='bold')

    # 1. Feature activation frequencies
    ax = axes[0][0]
    sorted_freq = np.sort(activation_freq)[::-1]
    colors_bar  = [COLORS['green'] if f > 0.001 else COLORS['red'] for f in sorted_freq]
    ax.bar(range(cfg_sae.d_sae), sorted_freq, color=colors_bar, alpha=0.85, width=1.0)
    ax.axhline(0.001, color=COLORS['orange'], linestyle='--', linewidth=1.5,
               label='0.1% threshold (dead/alive)')
    ax.legend(fontsize=8, facecolor=COLORS['panel'])
    styled_ax(ax, f'Feature Activation Frequencies\n({n_alive}/{cfg_sae.d_sae} alive)',
              'Feature (sorted)', 'P(active)')

    # 2. Decoder feature similarity (alive features only)
    ax = axes[0][1]
    alive_sim = similarity[alive_mask][:, alive_mask]
    im = ax.imshow(np.abs(alive_sim), cmap='viridis', vmin=0, vmax=1, aspect='auto')
    plt.colorbar(im, ax=ax, label='|cos similarity|')
    styled_ax(ax, f'Decoder Feature Similarities\n(alive features only, {n_alive})',
              'Feature j', 'Feature i')

    # 3. Best correlation with ground truth
    ax = axes[0][2]
    bins = np.linspace(0, 1, 20)
    ax.hist(best_gt_corr_val, bins=bins, color=COLORS['purple'], alpha=0.8, edgecolor='none')
    ax.axvline(0.5, color=COLORS['orange'], linestyle='--', linewidth=1.5, label='r=0.5')
    ax.axvline(0.8, color=COLORS['green'],  linestyle='--', linewidth=1.5, label='r=0.8')
    ax.legend(fontsize=8, facecolor=COLORS['panel'])
    styled_ax(ax, 'Max Correlation with GT Features\n(per alive SAE feature)',
              'Max |r| with any GT feature', 'Count')

    # 4. Reconstruction scatter
    ax = axes[1][0]
    n_plot = min(1000, acts_np.shape[0])
    for dim_idx in range(min(4, acts_np.shape[1])):
        ax.scatter(acts_np[:n_plot, dim_idx], x_hat_np[:n_plot, dim_idx],
                   alpha=0.15, s=6, label=f'dim {dim_idx}')
    lim = max(np.abs(acts_np[:n_plot]).max(), np.abs(x_hat_np[:n_plot]).max())
    ax.plot([-lim, lim], [-lim, lim], color=COLORS['red'], linewidth=1.5, linestyle='--', label='Perfect')
    ax.legend(fontsize=7, facecolor=COLORS['panel'], markerscale=2)
    styled_ax(ax, f'Reconstruction Quality\n(R² ≈ {variance_explained:.3f})',
              'True activation', 'Reconstructed activation')

    # 5. Decoder directions in 2D (if d_in == 2 or show first 2 dims)
    ax = axes[1][1]
    alive_W = W_dec_np[alive_mask, :2]  # Take first 2 hidden dims for visualization
    dead_W  = W_dec_np[~alive_mask, :2]
    ax.scatter(dead_W[:, 0],  dead_W[:, 1],  color=COLORS['red'],    alpha=0.3, s=15, label='Dead features')
    ax.scatter(alive_W[:, 0], alive_W[:, 1], color=COLORS['green'],  alpha=0.8, s=25, label='Alive features')
    # Draw unit circle
    theta = np.linspace(0, 2 * np.pi, 200)
    ax.plot(np.cos(theta), np.sin(theta), color=COLORS['orange'], linewidth=0.8, linestyle=':', alpha=0.5)
    ax.set_aspect('equal')
    ax.legend(fontsize=8, facecolor=COLORS['panel'])
    styled_ax(ax, 'Decoder Feature Directions (first 2 dims)',
              'W_dec[:, 0]', 'W_dec[:, 1]')

    # 6. Feature activation pattern on a batch
    ax = axes[1][2]
    sample_features = features_np[:50, alive_mask]
    im = ax.imshow(sample_features.T, aspect='auto', cmap='Blues', interpolation='nearest')
    plt.colorbar(im, ax=ax, label='Feature activation')
    styled_ax(ax, f'SAE Feature Activations (first 50 samples)\n(alive features only)',
              'Sample Index', 'SAE Feature Index')

    plt.tight_layout()
    plt.show()

    print('\n===== Feature Quality Summary =====')
    print(f'Total SAE features:       {cfg_sae.d_sae}')
    print(f'Alive features:           {n_alive} ({n_alive/cfg_sae.d_sae:.1%})')
    print(f'Dead features:            {cfg_sae.d_sae - n_alive}')
    print(f'Variance explained:       {variance_explained:.4f}')
    print(f'Mean recon error:         {recon_error:.6f}')
    print(f'Features with r>0.5 w/ GT: {(best_gt_corr_val > 0.5).sum()}/{n_alive}')
    print(f'Features with r>0.8 w/ GT: {(best_gt_corr_val > 0.8).sum()}/{n_alive}')

    return {
        'activation_freq':  activation_freq,
        'alive_mask':       alive_mask,
        'best_gt_corr_val': best_gt_corr_val,
        'variance_explained': variance_explained,
        'sae_gt_corr':      sae_gt_corr,
        'best_gt_match':    best_gt_match,
    }


analysis = analyze_learned_features(sae, hidden_acts, data, sae_cfg, cfg)

---
<a id='section-10'></a>
## 10. Part 5: The Loss Landscape — Sparsity vs. Reconstruction

The most critical hyperparameter in SAE training is `λ` (l1_coeff). Let's systematically study its effect.

In [ ]:
def sweep_lambda(lambdas, activations, cfg_sae, cfg_toy, n_epochs=800):
    """
    Train multiple SAEs with different L1 coefficients.
    Returns a list of result dicts for each lambda.
    """
    results = []

    for lam in lambdas:
        sweep_cfg = SAEConfig(
            d_in=cfg_sae.d_in,
            d_sae=cfg_sae.d_sae,
            l1_coeff=lam,
            lr=cfg_sae.lr,
            n_epochs=n_epochs,
            batch_size=cfg_sae.batch_size,
        )
        sweep_sae = SparseAutoencoder(sweep_cfg).to(device)

        # Quick training
        optimizer = optim.Adam(sweep_sae.parameters(), lr=sweep_cfg.lr)
        n_samples = activations.shape[0]
        n_batches = n_samples // sweep_cfg.batch_size

        for epoch in range(n_epochs):
            perm = torch.randperm(n_samples, device=device)
            for i in range(n_batches):
                idx = perm[i * sweep_cfg.batch_size:(i + 1) * sweep_cfg.batch_size]
                optimizer.zero_grad()
                out = sweep_sae(activations[idx])
                out['loss'].backward()
                optimizer.step()
                sweep_sae._normalize_decoder()

        sweep_sae.eval()
        with torch.no_grad():
            out = sweep_sae(activations)

        freq       = (out['f'] > 0).float().mean(0).cpu().numpy()
        recon_err  = out['recon_loss'].item()
        sparsity   = (out['f'] > 0).float().mean().item()
        n_alive    = (freq > 0.001).sum()

        results.append({
            'lambda':    lam,
            'recon_err': recon_err,
            'sparsity':  sparsity,
            'n_alive':   n_alive,
            'l1_loss':   out['l1_loss'].item(),
        })
        print(f'  λ={lam:.1e} | Recon: {recon_err:.5f} | Sparsity: {sparsity:.2%} | Alive: {n_alive}')

    return results


print('Sweeping over L1 coefficients (λ)...')
lambdas = [1e-5, 5e-5, 1e-4, 3e-4, 1e-3, 3e-3, 1e-2, 3e-2]
sweep_results = sweep_lambda(lambdas, hidden_acts, sae_cfg, cfg, n_epochs=600)
print('Done!')

In [ ]:
def plot_lambda_sweep(results):
    """Visualize the sparsity-reconstruction tradeoff."""
    lambdas   = [r['lambda']    for r in results]
    recon     = [r['recon_err'] for r in results]
    sparsity  = [r['sparsity']  for r in results]
    n_alive   = [r['n_alive']   for r in results]

    fig, axes = plt.subplots(1, 3, figsize=(16, 5), facecolor=COLORS['bg'])
    fig.suptitle('The Sparsity-Reconstruction Tradeoff (λ sweep)',
                 fontsize=13, color=COLORS['text'], fontweight='bold')

    # Plot 1: Reconstruction error vs lambda
    ax = axes[0]
    ax.plot(lambdas, recon, 'o-', color=COLORS['green'], linewidth=2, markersize=8)
    ax.set_xscale('log')
    ax.set_yscale('log')
    styled_ax(ax, 'Reconstruction Error vs λ', 'λ (l1_coeff)', 'MSE Reconstruction Error')

    # Plot 2: Sparsity vs lambda
    ax = axes[1]
    ax.plot(lambdas, sparsity, 'o-', color=COLORS['orange'], linewidth=2, markersize=8)
    ax.set_xscale('log')
    styled_ax(ax, 'Feature Sparsity vs λ\n(fraction of features active per sample)', 'λ', 'Fraction active')

    # Plot 3: Number of alive features vs lambda
    ax = axes[2]
    ax.plot(lambdas, n_alive, 'o-', color=COLORS['purple'], linewidth=2, markersize=8)
    ax.axhline(cfg.n_features, color=COLORS['blue'], linestyle='--', linewidth=1.5,
               label=f'Ground truth ({cfg.n_features} features)')
    ax.set_xscale('log')
    ax.legend(fontsize=8, facecolor=COLORS['panel'])
    styled_ax(ax, 'Alive Features vs λ', 'λ', 'Number of alive features')

    plt.tight_layout()
    plt.show()

    print('Key insight:')
    print('  Low λ  → Many alive features, good reconstruction, but not sparse (features overlap)')
    print('  High λ → Very sparse, but poor reconstruction (too much info lost)')
    print('  The sweet spot is where you have ~n_gt features alive with good reconstruction.')


plot_lambda_sweep(sweep_results)

---
<a id='section-11'></a>
## 11. Part 6: Dead Neurons & How to Fix Them

### The Dead Neuron Problem

A major practical challenge in SAE training is **dead neurons** (also called dead features). A neuron is dead if it **never activates** across the entire training set. This means:
- It contributes nothing to reconstruction
- It wastes capacity
- The gradient through a dead ReLU is always zero — it can't recover on its own

### Why Do Neurons Die?

1. **Initialization**: The encoder weight might point in a direction that is never positive after the bias is subtracted
2. **High L1**: Aggressively killing activations can send bias so negative neurons never recover
3. **Gradient flow collapse**: Once dead, the ReLU gradient is zero — the neuron gets no gradient signal

### The Fix: Neuron Resampling

The current best practice (from Anthropic's SAE papers) is periodic **neuron resampling**:
1. Identify dead neurons (haven't activated in the last N steps)
2. Re-initialize their encoder and decoder weights to a fresh random direction
3. Set their encoder bias to a value that makes them likely to activate

In [ ]:
class SAEWithResampling(SparseAutoencoder):
    """
    SAE with neuron resampling to fix dead neurons.

    Resampling strategy:
    - Every `resample_every` steps, identify dead neurons
    - For each dead neuron, sample a training point from the high-loss
      region of the dataset (where the SAE is making large errors)
    - Re-initialize the dead neuron's weights to point at that input
    - Reset its activation statistics
    """

    def __init__(self, cfg: SAEConfig, resample_every: int = 300):
        super().__init__(cfg)
        self.resample_every = resample_every
        self.steps = 0
        self.register_buffer('steps_since_active', torch.zeros(cfg.d_sae))

    def forward(self, x: torch.Tensor) -> dict:
        out = super().forward(x)
        f   = out['f']

        with torch.no_grad():
            self.steps_since_active += 1
            # Reset counter for neurons that just activated
            self.steps_since_active[f.sum(0) > 0] = 0

        self.steps += 1
        return out

    @torch.no_grad()
    def resample_dead_neurons(self, activations: torch.Tensor, n_dead_threshold: int = 50):
        """
        Resample neurons that have been dead for too long.

        We sample new directions from inputs where the SAE has high
        reconstruction error (these are "hard examples" that aren't
        being explained by current features).
        """
        dead_mask = self.steps_since_active > n_dead_threshold
        n_dead    = dead_mask.sum().item()

        if n_dead == 0:
            return 0

        # Find high-loss inputs
        out       = self.forward(activations[:2048])
        errors    = (activations[:2048] - out['x_hat']).pow(2).sum(dim=-1)  # (batch,)
        probs     = errors / errors.sum()
        probs_np  = probs.cpu().numpy()

        # Sample n_dead inputs proportional to their error
        n_sample  = min(n_dead, activations.shape[0])
        indices   = np.random.choice(len(probs_np), size=n_sample, replace=True, p=probs_np)
        new_dirs  = activations[indices] - self.b_dec  # (n_dead, d_in)

        # Normalize
        new_dirs  = F.normalize(new_dirs, dim=-1)      # unit norm

        # Get indices of dead neurons
        dead_idxs = dead_mask.nonzero(as_tuple=True)[0][:n_sample]

        # Re-initialize encoder weights for dead neurons
        self.W_enc[:, dead_idxs] = new_dirs.T
        self.b_enc[dead_idxs]    = 0.0

        # Re-initialize decoder weights
        self.W_dec[dead_idxs, :] = new_dirs

        # Reset their activation counters
        self.steps_since_active[dead_idxs] = 0

        return n_dead


def train_sae_with_resampling(
    activations: torch.Tensor,
    cfg_sae: SAEConfig,
    resample_every: int = 400,
) -> Tuple[SAEWithResampling, dict]:
    """Train SAE with periodic dead neuron resampling."""
    sae = SAEWithResampling(cfg_sae, resample_every=resample_every).to(device)
    optimizer = optim.Adam(sae.parameters(), lr=cfg_sae.lr)

    n_samples  = activations.shape[0]
    n_batches  = n_samples // cfg_sae.batch_size
    history    = defaultdict(list)
    step       = 0

    sae.train()
    for epoch in range(cfg_sae.n_epochs):
        perm = torch.randperm(n_samples, device=device)
        epoch_metrics = defaultdict(float)

        for i in range(n_batches):
            idx = perm[i * cfg_sae.batch_size:(i + 1) * cfg_sae.batch_size]
            x_batch = activations[idx]

            optimizer.zero_grad()
            out = sae(x_batch)
            out['loss'].backward()
            optimizer.step()
            sae._normalize_decoder()
            step += 1

            # Resample dead neurons periodically
            if step % resample_every == 0:
                n_resampled = sae.resample_dead_neurons(activations)

            epoch_metrics['loss']       += out['loss'].item()
            epoch_metrics['recon_loss'] += out['recon_loss'].item()
            epoch_metrics['sparsity']   += (out['f'] > 0).float().mean().item()

        for k, v in epoch_metrics.items():
            history[k].append(v / n_batches)

        dead_count = (sae.steps_since_active > 50).sum().item()
        history['dead_neurons'].append(dead_count)

        if epoch % 400 == 0:
            print(f'Epoch {epoch:5d} | Loss: {history["loss"][-1]:.5f} | '
                  f'Dead: {dead_count}/{cfg_sae.d_sae}')

    sae.eval()
    return sae, dict(history)


print('Training SAE with neuron resampling...')
sae_resamp, hist_resamp = train_sae_with_resampling(hidden_acts, sae_cfg)
print('Done!')

In [ ]:
# Compare dead neuron counts: with vs without resampling
fig, axes = plt.subplots(1, 2, figsize=(14, 5), facecolor=COLORS['bg'])
fig.suptitle('Effect of Neuron Resampling on Dead Features',
             fontsize=13, color=COLORS['text'], fontweight='bold')

ax = axes[0]
ax.plot(history['dead_neurons'],      color=COLORS['red'],   linewidth=2, label='Without resampling')
ax.plot(hist_resamp['dead_neurons'],  color=COLORS['green'], linewidth=2, label='With resampling')
ax.legend(fontsize=9, facecolor=COLORS['panel'])
styled_ax(ax, 'Dead Neurons Over Training', 'Epoch', f'Dead neurons / {sae_cfg.d_sae}')

ax = axes[1]
ax.plot(history['recon_loss'],     color=COLORS['red'],   linewidth=2, label='Without resampling')
ax.plot(hist_resamp['recon_loss'], color=COLORS['green'], linewidth=2, label='With resampling')
ax.set_yscale('log')
ax.legend(fontsize=9, facecolor=COLORS['panel'])
styled_ax(ax, 'Reconstruction Loss Comparison', 'Epoch', 'MSE (log scale)')

plt.tight_layout()
plt.show()

print(f'Final dead neurons without resampling: {history["dead_neurons"][-1]}')
print(f'Final dead neurons with resampling:    {hist_resamp["dead_neurons"][-1]}')

---
<a id='section-12'></a>
## 12. How SAEs Are Used in Real Research

### From Toy Models to Real LLMs

The exact same architecture works on real language models. Here's how researchers use SAEs:

#### Step 1: Collect Activations
Run a large corpus of text through the model and save the activations at a specific layer (e.g., the residual stream after MLP layer 6).

#### Step 2: Train the SAE
Train an SAE on these activations. The SAE dimensions are now much larger:
- `d_in` ≈ 4096 (GPT-4 class models)
- `d_sae` ≈ 65,536 (16x expansion)

#### Step 3: Interpret Features
For each alive SAE feature:
1. Find the **top-activating tokens** — what inputs make this feature fire most?
2. **Intervene** — artificially set this feature to high/zero and observe what changes in model output
3. **Label** the feature: e.g., "Feature 12,441 activates on DNA sequences"

#### Step 4: Build Feature Circuits
Trace how features interact across layers to understand how computations happen.

### What Has Been Found

Anthropic's team trained SAEs on Claude and GPT-2 and found features including:
- **Concrete features**: "The Great Wall of China", "blood types", "genetic recombination"
- **Safety-relevant features**: A "Assistant" token feature that, when suppressed, made Claude refuse instructions
- **Reasoning features**: Features that activate during multi-step problem solving

### The Tools You Will Use (Soon)

```python
# TransformerLens: extract LLM activations
import transformer_lens
model = transformer_lens.HookedTransformer.from_pretrained('gpt2')
_, cache = model.run_with_cache("The capital of France is")
activations = cache['resid_post', 6]  # Layer 6 residual stream

# SAELens: pre-trained SAEs for popular models
from sae_lens import SAE
sae, cfg_dict, _ = SAE.from_pretrained(
    release='gpt2-small-res-jb',
    sae_id='blocks.6.hook_resid_pre'
)
features = sae.encode(activations)
```

### Evaluation Metrics Used in Research

| Metric | What it measures | Target |
|--------|-----------------|--------|
| **L0** (mean features active) | Sparsity — how many features per token | ~20-50 |
| **CE Loss Increase** | How much does replacing activations with SAE reconstruction hurt model performance? | < 0.05 nats |
| **Variance Explained** | What fraction of activation variance is captured? | > 95% |
| **Dead Features** | Fraction of features never used | < 5% |
| **Feature Monosemanticity** | Do features correspond to single concepts? | Qualitative |

---
<a id='section-13'></a>
## 13. Limitations & Open Problems

SAEs are powerful but far from complete. Here are the major open research questions:

### 1. The Evaluation Problem
How do you know if an SAE feature is a "real" feature vs. a spurious artifact? There's no ground truth for real LLMs. Current approaches:
- **Activation patching**: Replace features and measure behavioral changes
- **Human labeling**: Do humans agree on what a feature represents?
- **Automated interpretability**: Use another LLM to label features

### 2. Computational Scale
Training an SAE on a large model's activations requires:
- Collecting billions of activation vectors
- Training an SAE with millions of features
- Analyzing the features at scale

This is expensive. Anthropic's recent SAE on Claude 3 Sonnet had **34 million features**.

### 3. Feature Splitting & Absorption
With more SAE capacity (larger d_sae), features can **split**: what was one feature becomes multiple more specific sub-features. This raises questions about the right level of granularity.

### 4. The Universality Hypothesis
Do different models learn the same features? Early evidence suggests yes — similar circuits emerge in GPT-2, Claude, LLaMA — but this isn't well understood.

### 5. Beyond the Residual Stream
Most SAE work focuses on residual stream activations. What about attention patterns, MLP neurons, key/query/value vectors?

### 6. Superposition-Free Training
Can we train LLMs with an architecture that *prevents* superposition from forming in the first place? This remains an open question.

### 7. Gated SAEs & TopK SAEs
Recent variants include:
- **Gated SAEs**: A separate gate network decides which features to activate, improving L0/reconstruction tradeoff
- **TopK SAEs**: Instead of L1 penalty, directly fix the number of active features to K (TopK operation)

In [ ]:
# BONUS: TopK SAE — an alternative to L1 sparsity

class TopKSAE(nn.Module):
    """
    TopK Sparse Autoencoder.

    Instead of using an L1 penalty to encourage sparsity, this variant
    directly enforces exactly K active features per sample using the
    TopK operation. This gives exact control over sparsity without
    needing to tune λ.

    Introduced in: "Scaling and evaluating sparse autoencoders"
    (Gao et al., Anthropic 2024).

    Loss: Pure reconstruction loss (no L1 term needed!)
        L = ||x - x̂||²
    where x̂ is computed from only the top K features.
    """

    def __init__(self, d_in: int, d_sae: int, k: int):
        super().__init__()
        self.d_in  = d_in
        self.d_sae = d_sae
        self.k     = k

        self.W_enc = nn.Parameter(torch.randn(d_in, d_sae) * 0.01)
        self.b_enc = nn.Parameter(torch.zeros(d_sae))
        self.W_dec = nn.Parameter(torch.randn(d_sae, d_in) * 0.01)
        self.b_dec = nn.Parameter(torch.zeros(d_in))

        self._normalize_decoder()

    @torch.no_grad()
    def _normalize_decoder(self):
        norms = self.W_dec.norm(dim=1, keepdim=True).clamp(min=1e-8)
        self.W_dec.data = self.W_dec.data / norms

    def encode(self, x: torch.Tensor) -> torch.Tensor:
        pre_act = (x - self.b_dec) @ self.W_enc + self.b_enc
        # TopK: zero out all but the top K activations
        topk_vals, topk_idx = torch.topk(pre_act, self.k, dim=-1)
        f = torch.zeros_like(pre_act)
        f.scatter_(-1, topk_idx, F.relu(topk_vals))  # ReLU for non-negativity
        return f

    def forward(self, x: torch.Tensor) -> dict:
        f     = self.encode(x)
        x_hat = f @ self.W_dec + self.b_dec
        loss  = (x - x_hat).pow(2).sum(-1).mean()  # Pure reconstruction, no L1!
        return {'loss': loss, 'f': f, 'x_hat': x_hat}


# Train and compare
k_value = 3  # Exactly 3 features active per sample
topk_sae = TopKSAE(d_in=sae_cfg.d_in, d_sae=sae_cfg.d_sae, k=k_value).to(device)

optimizer_topk = optim.Adam(topk_sae.parameters(), lr=2e-3)
n_samples = hidden_acts.shape[0]
n_batches = n_samples // sae_cfg.batch_size
topk_losses = []

topk_sae.train()
for epoch in range(1500):
    perm = torch.randperm(n_samples, device=device)
    epoch_loss = 0.0
    for i in range(n_batches):
        idx = perm[i * sae_cfg.batch_size:(i + 1) * sae_cfg.batch_size]
        optimizer_topk.zero_grad()
        out = topk_sae(hidden_acts[idx])
        out['loss'].backward()
        optimizer_topk.step()
        topk_sae._normalize_decoder()
        epoch_loss += out['loss'].item()
    topk_losses.append(epoch_loss / n_batches)
    if epoch % 500 == 0:
        print(f'TopK SAE | Epoch {epoch:5d} | Loss: {topk_losses[-1]:.5f}')

topk_sae.eval()
with torch.no_grad():
    out_topk = topk_sae(hidden_acts)

mean_active = (out_topk['f'] > 0).float().mean().item()
print(f'\nTopK SAE (k={k_value}) results:')
print(f'  Reconstruction loss: {out_topk["loss"].item():.5f}')
print(f'  Mean features active: {out_topk["f"].sum(-1).mean().item():.2f} (exactly {k_value} per sample)')
print(f'  Alive features: {(out_topk["f"].sum(0) > 0).sum().item()}/{sae_cfg.d_sae}')

---
<a id='section-14'></a>
## 14. Further Reading & Key Resources

### 📄 Foundational Papers (Read in This Order)

1. **Toy Models of Superposition** (Elhage et al., Anthropic, 2022)  
   The paper that started everything. Proves superposition exists and builds intuition.  
   🔗 https://transformer-circuits.pub/2022/toy_model/index.html

2. **Towards Monosemanticity: Decomposing Language Models with Dictionary Learning** (Bricken et al., Anthropic, 2023)  
   The first paper applying SAEs to a real (one-layer) transformer. Finds beautiful, interpretable features.  
   🔗 https://transformer-circuits.pub/2023/monosemantic-features/index.html

3. **Scaling and Evaluating Sparse Autoencoders** (Gao et al., Anthropic, 2024)  
   Introduces TopK SAEs, scaling laws for SAEs, rigorous evaluation metrics.  
   🔗 https://arxiv.org/abs/2406.04093

4. **Improving Dictionary Learning with Gated Sparse Autoencoders** (Rajamanoharan et al., DeepMind, 2024)  
   Introduces Gated SAEs which improve the sparsity-reconstruction tradeoff.  
   🔗 https://arxiv.org/abs/2404.16014

5. **Mapping the Mind of a Large Language Model** (Templeton et al., Anthropic, 2024)  
   SAEs applied to Claude 3 Sonnet. Found 34 million features. Landmark paper.  
   🔗 https://www.anthropic.com/research/mapping-mind-language-model

---

### 🎓 Courses & Tutorials

6. **ARENA Curriculum (Chapter 1: Transformers & Interp)**  
   The best structured curriculum for mech interp. Hands-on, with exercises.  
   🔗 https://arena-chapter1-transformer-interp.streamlit.app/

7. **Neel Nanda's 200 Concrete Open Problems in Mech Interp**  
   A guide to what the field is working on. Essential for knowing where to contribute.  
   🔗 https://www.alignmentforum.org/posts/LbrPTJ4fmABEdEnLf/200-concrete-open-problems-in-mechanistic-interpretability

8. **TransformerLens Library Documentation**  
   The main tool for extracting LLM activations for mech interp research.  
   🔗 https://transformerlensorg.github.io/TransformerLens/

9. **SAELens Documentation** (Joseph Bloom)  
   Pre-trained SAEs for popular models, ready to use.  
   🔗 https://jbloomaus.github.io/SAELens/

10. **Neuronpedia** — Visual browser for LLM features  
    Browse 100,000s of discovered features, see their top activating examples.  
    🔗 https://www.neuronpedia.org/

---

### 📝 Blog Posts & Explainers

11. **A Mathematical Framework for Transformer Circuits** (Elhage et al., 2021)  
    The conceptual foundation for understanding what LLMs compute.  
    🔗 https://transformer-circuits.pub/2021/framework/index.html

12. **Neel Nanda's SAE Tutorial** (Colab notebook)  
    Step-by-step SAE training on GPT-2 with TransformerLens + SAELens.  
    🔗 https://colab.research.google.com/drive/1ZgS5bPXNROFmgBsFkM6GDCBhXJNwbGxH

13. **Sparse Dictionary Learning (Wikipedia)**  
    The classical signal processing foundation that SAEs are built on.  
    🔗 https://en.wikipedia.org/wiki/Sparse_dictionary_learning

14. **Adam Jermyn's Thread on SAE Intuitions** (Twitter/X)  
    Excellent intuitive explanations of why SAEs work.  
    🔗 https://x.com/AdamJermyn

---

### 🛠️ Tools to Install and Explore Next

```bash
pip install transformer_lens   # Extract LLM activations
pip install sae_lens           # Pre-trained SAEs for GPT-2, Mistral, Gemma, etc.
pip install circuitsvis        # Visualize attention patterns
```

### 🗺️ Your Learning Path from Here

| Stage | Goal | Resource |
|-------|------|----------|
| **Now** | Understand superposition + SAE math | This notebook ✓ |
| **Next** | Run a pre-trained SAE on GPT-2 | SAELens + Neel's Colab |
| **Then** | Understand transformer circuits | ARENA Ch. 1 |
| **Later** | Train SAE on a real model layer | SAELens training script |
| **Research** | Identify novel interpretable features | Your own contribution! |

In [ ]:
# ── Final experiment: Everything at once ─────────────────────────
# Let's do one final clean visualization summarizing what we learned

print('=== SPARSE AUTOENCODER — COMPLETE SUMMARY ===')
print()
print('THE PROBLEM: Superposition')
print(f'  A {cfg.n_features}-feature world compressed into {cfg.d_hidden} dimensions.')
print(f'  Individual neurons are polysemantic — they encode multiple features.')
print()
print('THE SOLUTION: Sparse Autoencoders')
print(f'  Map {sae_cfg.d_in}-dim activations → {sae_cfg.d_sae}-dim sparse features.')
print(f'  Expansion factor: {sae_cfg.d_sae // sae_cfg.d_in}x (more slots than original features)')
print()
print('THE MATH:')
print('  f = ReLU(W_enc @ (x - b_dec) + b_enc)   [encoder, sparse]')
print('  x̂ = W_dec @ f + b_dec                   [decoder, reconstruction]')
print('  L = ||x - x̂||² + λ||f||₁               [loss: recon + sparsity]')
print()
print('TRAINING RESULTS:')

sae.eval()
with torch.no_grad():
    final_out = sae(hidden_acts)
    final_freq = (final_out['f'] > 0).float().mean(0)
    n_alive_final = (final_freq > 0.001).sum().item()
    recon_final = final_out['recon_loss'].item()
    sparsity_final = (final_out['f'] > 0).float().mean().item()

print(f'  Alive features:    {n_alive_final}/{sae_cfg.d_sae} ({n_alive_final/sae_cfg.d_sae:.1%})')
print(f'  Reconstruction MSE:{recon_final:.5f}')
print(f'  Sparsity:          {sparsity_final:.2%} features active per sample')
print(f'  Ground truth:      {cfg.n_features} features to recover')
print()
print('KEY INSIGHTS:')
print('  1. Superposition is WHY interpretability is hard')
print('  2. SAEs decompress superimposed features into interpretable components')
print('  3. The L1 penalty (or TopK) enforces sparsity, making features monosemantic')
print('  4. Dead neurons are a real problem — resampling helps significantly')
print('  5. λ (l1_coeff) controls sparsity-reconstruction tradeoff — must be tuned')
print()
print('NEXT STEP: Apply this to real LLM activations using TransformerLens + SAELens!')